In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(81426, 31)


In [2]:
interaction = (
    df.groupby(
        ["customerId", "skuNumber"]
    )["effective_qty"]
    .sum()
    .reset_index()
)

interaction.head()

,customerId,skuNumber,effective_qty
0,USR-100,SKU00010,1
1,USR-100,SKU00030,1
2,USR-100,SKU00060,2
3,USR-100,SKU00062,1
4,USR-100,SKU00084,1


In [3]:
print(
    "Interactions:",
    len(interaction)
)

print(
    "Retailers:",
    interaction["customerId"].nunique()
)

print(
    "Products:",
    interaction["skuNumber"].nunique()
)

Interactions: 39282
Retailers: 5421
Products: 197


In [4]:
matrix = interaction.pivot(
    index="customerId",
    columns="skuNumber",
    values="effective_qty"
).fillna(0)

matrix.shape

(5421, 197)

In [5]:
zeros = (matrix == 0).sum().sum()

total = (
    matrix.shape[0]
    * matrix.shape[1]
)

sparsity = (
    zeros / total
) * 100

print(
    f"Sparsity: {sparsity:.2f}%"
)

Sparsity: 96.32%


In [6]:
matrix.to_parquet(
    "../data/processed/interaction_matrix.parquet"
)

print("Saved")

Saved


In [7]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category"
        ]
    ]
    .drop_duplicates()
)

sku_lookup.to_parquet(
    "../data/processed/sku_lookup.parquet",
    index=False
)

print("Saved")

Saved


In [8]:
top_products = (
    df.groupby("itemName")
    ["effective_qty"]
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
)

top_products

itemName
Rajnigandha 17g Zipper 1 Pcs             26296
Rajnigandha 4g                           22999
TZ 00 (with Silver) 0.45g                22960
TZ 00 4.25g (6 Pcs)                       8769
Rajnigandha 100g                          5532
RG Pearl Elaichi MP Pouch 0.14g           3442
RG Pearl Elaichi 1.4g Hanger              2730
Rajnigandha 17g Zipper IC                 2638
Pass Pass Meetha Magic 11.75g Hanger      2152
Pass Pass Meetha Magic 2.2g (100 pcs)     2080
Center Fresh                              1540
Pulse Kachha Aam 175 Candy                1398
Bingo Tedhe Medhe                         1331
Center Fruit                              1289
TZ 00 10g TIN                             1280
Jabsons Hing Jeera Peanuts                1122
Alpenliebe Creamfills Butter Toffee       1029
RG Pearl Elaichi Hanger (6 pcs)            967
Hell Energy Drink                          946
Tulsi Royal Khajoor                        876
Name: effective_qty, dtype: int64

In [9]:
matrix.shape

(5421, 197)

In [10]:
top_products.head(10)

itemName
Rajnigandha 17g Zipper 1 Pcs             26296
Rajnigandha 4g                           22999
TZ 00 (with Silver) 0.45g                22960
TZ 00 4.25g (6 Pcs)                       8769
Rajnigandha 100g                          5532
RG Pearl Elaichi MP Pouch 0.14g           3442
RG Pearl Elaichi 1.4g Hanger              2730
Rajnigandha 17g Zipper IC                 2638
Pass Pass Meetha Magic 11.75g Hanger      2152
Pass Pass Meetha Magic 2.2g (100 pcs)     2080
Name: effective_qty, dtype: int64

In [11]:
matrix.shape

(5421, 197)